# Project Schedule Optimization and Interpretation with OptaPy and SHAP

This notebook demonstrates how to optimize a project schedule with 100 sample tasks using OptaPy and interprets model predictions with SHAP. We'll use machine learning to predict task durations, feed predictions into OptaPy for optimization, and then analyze the impact of various features on scheduling using SHAP.

## 1. Importing Libraries and Generating Sample Data


In [30]:
from optapy import solver_factory_create
from optapy.types import SolverConfig, Duration
from optapy.score import HardSoftScore
from optapy import planning_entity, planning_solution, planning_variable, planning_entity_collection_property, \
    value_range_provider, constraint_provider
from org.optaplanner.core.api.domain.variable import PlanningVariableGraphType

# Define the Task entity with necessary attributes
@planning_entity
class Task:
    def __init__(self, task_id, duration, priority, dependencies=None, start_time=None):
        self.task_id = task_id
        self.duration = duration
        self.priority = priority
        self.dependencies = dependencies or []  # Tasks that must be completed before this one
        self.start_time = start_time

    @planning_variable(value_range_provider_refs=['timeRange'], variable_type=PlanningVariableGraphType.NONE)
    def get_start_time(self):
        return self.start_time

    def set_start_time(self, start_time):
        self.start_time = start_time

@planning_solution
class Schedule:
    def __init__(self, tasks, time_slots):
        self._tasks = tasks  # Private attribute for tasks
        self._time_slots = time_slots  # Private attribute for time slots

    # Getter for tasks annotated as a planning entity collection property
    @planning_entity_collection_property
    def get_tasks(self):
        return self._tasks

    # Getter for time slots annotated as a value range provider
    @value_range_provider
    def get_time_slots(self):
        return self._time_slots

# Define constraints for the project schedule
@constraint_provider
def define_constraints(constraint_factory):
    # Prevent overlapping tasks for the same resource
    return []

# Configure the Solver
solver_config = SolverConfig() \
    .withEntityClasses(Task) \
    .withSolutionClass(Schedule) \
    .withConstraintProviderClass(define_constraints) \
    .withTerminationSpentLimit(Duration.ofSeconds(30))

# Create the Solver
solver = solver_factory_create(solver_config).buildSolver()

# Define sample data for tasks and time slots
def generate_schedule_problem():
    tasks = [
        Task(task_id=1, duration=3, priority=1, dependencies=[]),
        Task(task_id=2, duration=2, priority=2, dependencies=[1]),
    ]
    time_slots = list(range(10))  # Example range of possible start times
    return Schedule(tasks, time_slots)

# Solve the schedule optimization problem
solution = solver.solve(generate_schedule_problem())

# Output the solution
for task in solution.get_tasks():
    print(f"Task {task.task_id} scheduled to start at time {task.start_time} with duration {task.duration}")


java.lang.IllegalStateException: java.lang.IllegalStateException: The solutionClass (class org.jpyinterpreter.user.Schedule$$4) must have at least 1 member with a PlanningEntityCollectionProperty annotation or a PlanningEntityProperty annotation.

In [16]:
from optapy import solver_factory_create
from optapy.types import SolverConfig, Duration

# Define your SolverConfig and constraints as per your requirements
solver_config = SolverConfig()
# Add additional configurations if needed

solver = solver_factory_create(solver_config).buildSolver()


java.lang.IllegalArgumentException: java.lang.IllegalArgumentException: The solver configuration must have a solutionClass (null). If you're using the Quarkus extension or Spring Boot starter, it should have been filled in already.

### Generating a Sample Dataset of Tasks

Each task has several features:
- `task_id`: Unique identifier for each task
- `resource`: Type of resource required
- `duration`: Predicted duration in hours (target variable)
- `complexity`: Complexity rating (1 to 5)
- `dependency_count`: Number of dependencies
- `priority`: Priority level (1 to 3)


In [4]:
# Code Cell: Generate sample data for 100 tasks
n_tasks = 100
tasks = pd.DataFrame({
    'task_id': range(1, n_tasks + 1),
    'resource': np.random.choice(['Team A', 'Team B', 'Team C'], n_tasks),
    'duration': np.random.randint(1, 10, n_tasks),
    'complexity': np.random.randint(1, 6, n_tasks),
    'dependency_count': np.random.randint(0, 4, n_tasks),
    'priority': np.random.randint(1, 4, n_tasks)
})

# Display first few rows
tasks.head()


,task_id,resource,duration,complexity,dependency_count,priority
0,1,Team B,6,4,3,1
1,2,Team B,2,4,2,1
2,3,Team A,5,2,0,1
3,4,Team B,6,1,2,1
4,5,Team B,4,3,3,2


## 2. Predicting Task Durations with Machine Learning

We will train a `RandomForestRegressor` model to predict `duration` based on other task features. The model will help simulate a real-world situation where machine learning informs scheduling.


In [5]:
# Code Cell: Prepare data for model training
X = tasks[['complexity', 'dependency_count', 'priority']]
y = tasks['duration']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


NameError: name 'RandomForestRegressor' is not defined

## 3. Interpreting Model Predictions with SHAP

Using SHAP, we can understand feature importance for the predicted `duration`. SHAP values indicate how each feature impacts individual predictions.


In [9]:
# Code Cell: Apply SHAP to understand feature contributions
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Visualize feature importance for task duration predictions
shap.summary_plot(shap_values, X_test)


XGBoostError: [01:17:56] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\c_api\c_api_utils.h:129: Check failed: std::accumulate(shape.cbegin(), shape.cend(), static_cast<bst_ulong>(1), std::multiplies<>{}) == chunksize * rows (80 vs. 340) : 

## 4. Optimization of Project Schedule with OptaPy

Now we define our scheduling constraints and use OptaPy to optimize the schedule.


In [8]:
# Code Cell: Define OptaPy scheduling model and constraints (mock example for illustration)

# Task entity definition for OptaPy
class Task:
    def __init__(self, task_id, resource, duration, complexity, dependency_count, priority):
        self.task_id = task_id
        self.resource = resource
        self.duration = duration
        self.complexity = complexity
        self.dependency_count = dependency_count
        self.priority = priority
        self.start_time = None

# Create Task instances based on the dataset
task_objects = [Task(row['task_id'], row['resource'], row['duration'], 
                     row['complexity'], row['dependency_count'], row['priority']) for _, row in tasks.iterrows()]

# Define a simple constraint: No overlapping tasks for the same resource
def no_overlapping_tasks(constraint_factory):
    return constraint_factory.forEach(Task).filter(lambda task: task.resource).penalize("No overlapping tasks", 1)

# Define and configure solver
solver_factory = SolverFactory.create(solver_config_xml="""
<solver>
    <solutionClass>Task</solutionClass>
    <scoreDirectorFactory>
        <constraintProviderClass>no_overlapping_tasks</constraintProviderClass>
    </scoreDirectorFactory>
    <termination>
        <secondsSpentLimit>10</secondsSpentLimit>
    </termination>
</solver>
""")
solver = solver_factory.buildSolver()
solution = solver.solve(task_objects)

# Output solution
solution


NameError: name 'SolverFactory' is not defined

## 5. Sensitivity Analysis on Constraints and Features

Conduct sensitivity analysis to understand the impact of key features and constraints on the optimized schedule. This step mimics SHAP by analyzing "what-if" scenarios on constraints.


## Conclusion

In this notebook, we demonstrated a workflow for project schedule optimization using OptaPy and SHAP. We used SHAP to interpret machine learning predictions and OptaPy to optimize the scheduling of tasks based on constraints. This combined approach allows for both optimized scheduling and interpretability, supporting better decision-making for complex projects.
